<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Introduction to Spark and PySpark

*Session 5 · Notebook 01.02 · Lecture · Coach version*

## Overview

In Sessions 2 to 4 everything ran in **pandas and scikit-learn on a single machine**. That works until the data no longer fits in one computer's memory, or a job takes hours because one processor does all the work. **Apache Spark** is the standard answer: an engine that spreads both the data and the computation across a cluster of machines. **PySpark** is its Python interface.

The notebook has two parts:

- **Part 1: Spark and PySpark.** What Spark is, how a Spark application is organised, what makes it different from pandas (above all, **lazy evaluation**), and how to look at the Spark session Databricks gives you and its configuration.
- **Part 2: The pandas walkthrough in PySpark.** The Session 2 **pandas walkthrough**, step by step, on the same data (the **Pagila** `customer` and `payment` tables, three years of **Superstore orders** and the **sales representatives**), so that for every step you see the pandas line you already know next to the PySpark line that does the same job on a cluster. Along the way you meet **schemas**, **immutability** (no `inplace=True`), the absence of an **index**, why custom Python functions are the last resort rather than the first, and the **pandas API on Spark**, which runs pandas syntax on Spark.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what Apache Spark and PySpark are, and the problems they solve.
- Describe the parts of a Spark application: driver, executors, cluster manager, partitions and tasks.
- Explain transformations, actions and lazy evaluation, and decide when Spark is the right tool and when pandas is enough.
- Use the `SparkSession`, read its version and configuration settings, and explain what the main settings control.
- Read CSV files into Spark DataFrames, and define a **schema** so that every column has the right type.
- Explore, select, rename (with aliases, without SQL), sort, create and drop columns, and filter rows in several ways.
- Use the PySpark equivalents of pandas `.apply()` (built-in functions first, a Python UDF when there is no other way).
- Write results to files, stack DataFrames with `unionByName`, combine them with `join` (also when the key has different names), aggregate with `groupBy().agg()`, and run the same query in Spark SQL.
- Use the pandas API on Spark to run pandas-style code on Spark, and know its limits.

## Prerequisites

- pandas from Sessions 2 and 3, and basic SQL. Every section of Part 2 has a twin in the Session 2 pandas walkthrough.
- Optional: notebook 01.00 (the environment pre-flight check), if your trainer asks you to run it.

## Index

**Part 1: Spark and PySpark**

1. [Why this matters](#why)
2. [What is Apache Spark?](#spark)
3. [How a Spark application is organised](#architecture)
4. [The Spark ecosystem](#ecosystem)
5. [What is PySpark?](#pyspark)
6. [Lazy evaluation: transformations and actions](#lazy)
7. [Spark or pandas?](#when)
8. [Meet your Spark session](#first)

**Part 2: The pandas walkthrough in PySpark**

9. [From pandas to PySpark](#frompandas)
10. [Reading data](#setup)
11. [Schemas: defining the structure of your data](#schema)
12. [Exploring the data](#explore)
13. [Selecting and renaming columns](#select)
14. [Sorting, and why there is no index](#sort)
15. [Creating columns, and the `.apply()` equivalents](#create)
16. [Removing columns](#drop)
17. [Filtering rows](#filter)
18. [Writing data to files](#write)
19. [Working with multiple datasets: unions and joins](#multiple)
20. [Aggregating data](#agg)
21. [The same in Spark SQL](#sql)
22. [The pandas API on Spark](#pandasapi)
23. [Application: a sales report from three years of orders](#application)
24. [Exercises](#exercises)
25. [Additional Exercises](#additional)
26. [Challenge](#challenge)
27. [Key Takeaways](#takeaways)
28. [Conclusion](#conclusion)
29. [Further Reading & Resources](#reading)

# Part 1: Spark and PySpark

<a id="why"></a>
# Section 1: Why this matters

Data volumes keep growing: every card transaction, every log line, every sensor reading, kept for years. Two limits appear as soon as a dataset outgrows a laptop:

| Limit | What you see | What Spark does |
|---|---|---|
| **Memory** | pandas raises `MemoryError` or the machine grinds to a halt | Splits the data into partitions spread across many machines |
| **Time** | One processor works through billions of rows one after another | Processes all partitions in parallel |
| **Reliability** | A crash halfway through a long job means starting again | Recomputes only the lost pieces, from the recorded steps |
| **Skills** | Big-data tools used to require Java and new concepts | DataFrames and SQL that look like pandas and SQL you already know |

Spark is also the engine underneath **Databricks**, where this session runs. Learning PySpark therefore means learning the language of the platform many organisations now use for analytics, reporting and machine learning at scale.

<a id="spark"></a>
# Section 2: What is Apache Spark?

**Definition.** **Apache Spark** is an open-source engine for processing large datasets in parallel across a cluster of computers. It keeps intermediate results **in memory** wherever possible, and it offers high-level APIs in Python, SQL, Scala, Java and R.

**Example.** To compute the average transaction value over five billion rows, Spark splits the rows into thousands of chunks, has many machines average their chunks at the same time, and combines the partial results into one answer.

**Analogy.** pandas is a single chef cooking a banquet alone. Spark is a head chef running a kitchen brigade: the head chef plans the menu and hands out the work, each cook prepares part of it at the same time, and the dishes come together at the pass.

**Explanation.** Spark was created at UC Berkeley around 2009 as a faster successor to **Hadoop MapReduce**, which wrote every intermediate step to disk. Its design goals explain much of how it behaves:

| Goal | How Spark achieves it |
|---|---|
| **Speed** | In-memory processing and a query optimiser that plans the whole job before running it |
| **Ease of use** | High-level DataFrame and SQL APIs instead of low-level map and reduce code |
| **Modularity** | One engine for SQL, machine learning and streaming, with the same DataFrames throughout |
| **Extensibility** | Reads and writes many sources: CSV, JSON, Parquet, Delta tables, databases, cloud storage |

Internally, Spark represents a job as a **DAG** (directed acyclic graph): the steps to perform and how they depend on each other. Having the whole graph before running anything lets it reorder and combine steps for efficiency, and recompute a lost partition by replaying only the steps that produced it.

<a id="architecture"></a>
# Section 3: How a Spark application is organised

**Definition.** Every Spark application has one **driver** and several **executors**, coordinated by a **cluster manager**.

<center><img src="https://spark.apache.org/docs/latest/img/cluster-overview.png" width="560" alt="Spark cluster overview: driver program, cluster manager, worker nodes with executors" /></center>

*Source: [Apache Spark cluster overview](https://spark.apache.org/docs/latest/cluster-overview.html)*

**Explanation.**

| Term | What it is | In the kitchen analogy |
|---|---|---|
| **Driver** | The process running your notebook code. It builds the plan, splits it into tasks and collects results | The head chef |
| **Cluster manager** | Allocates machines and resources (on Databricks this is managed for you) | The restaurant manager who assigns staff |
| **Worker node** | A machine in the cluster | A kitchen station |
| **Executor** | A process on a worker that runs tasks and holds data in memory | A cook at a station |
| **Partition** | A chunk of the data; the unit of parallelism | One tray of ingredients |
| **Task** | The work for one partition in one step | Preparing one tray |
| **Stage** | A group of tasks that can run without exchanging data | One course, prepared in parallel |

Two consequences you will meet again:

- Your code runs on the **driver**, but the data lives on the **executors**. Bringing all of it back to the driver (with `collect()` or `toPandas()`) can overwhelm that one machine.
- Some steps (filters, new columns) work on each partition independently. Others (grouping, joining, sorting) must **shuffle** rows between executors so that matching rows meet; shuffles are the expensive part (notebook 02.04).

**On Databricks** the cluster, cluster manager and `SparkSession` are created for you; you attach a notebook to compute and start writing code.

<a id="ecosystem"></a>
# Section 4: The Spark ecosystem

Spark is one engine with several libraries on top, all sharing the same DataFrames:

| Library | Purpose |
|---|---|
| **Spark Core** | Task scheduling, memory management, the low-level RDD API |
| **Spark SQL and DataFrames** | Structured data with SQL or a DataFrame API, and the Catalyst query optimiser |
| **MLlib** (`pyspark.ml`) | Scalable machine learning: feature transformers, models, pipelines, evaluation |
| **Structured Streaming** | The same DataFrame code applied to continuously arriving data |
| **GraphX / GraphFrames** | Graph processing |

**What we cover:** Spark SQL and DataFrames (reading, exploring, cleaning, joining and aggregating data, and making it run fast) and MLlib (building, evaluating and tuning machine learning models). Spark Core's RDD API is only mentioned (Section 5); streaming and graph processing are not covered.

Because the libraries share one engine, a single job can read raw files, clean them with SQL, and train a model with MLlib, without moving the data to another system.

<a id="pyspark"></a>
# Section 5: What is PySpark?

**Definition.** **PySpark** is the Python API for Spark. You write Python; Spark's engine (which runs on the Java Virtual Machine, the JVM) does the distributed work.

**Analogy.** A conductor and an orchestra. You (the Python code) give instructions; the orchestra (the JVM engine across the cluster) plays. You never touch the instruments directly, which is why built-in Spark functions are fast: the work stays inside the engine.

**Explanation.** PySpark talks to the JVM through a bridge library called Py4J. When you call `df.filter(...)` you are not filtering in Python; you are adding a step to the engine's plan. This is why **Java is required** to run Spark on a laptop (Databricks provides it), and why custom Python functions applied row by row (UDFs) are slow: each row must cross the bridge (notebook 02.04).

PySpark offers several ways to work, all compiling to the same engine:

| Style | Looks like | Example |
|---|---|---|
| DataFrame API | pandas-like method chains | `df.filter(F.col("age") > 30).select("name")` |
| Spark SQL | SQL strings | `spark.sql("SELECT name FROM people WHERE age > 30")` |
| RDD API | Low-level functions on Python objects | `rdd.filter(lambda r: r[1] > 30)` |
| pandas API on Spark | pandas syntax on Spark (`pyspark.pandas`) | `psdf[psdf.age > 30]["name"]` |

We focus on the **DataFrame API and SQL**, the standard for modern PySpark; Section 22 shows the pandas API on Spark.

> **Good to know: RDDs.** The **RDD** (Resilient Distributed Dataset) is Spark's original, low-level data structure: a distributed collection of Python objects processed with your own functions, often short lambdas such as `rdd.map(lambda x: x * 2)`. DataFrames are built on top of RDDs and add named, typed columns, which lets Spark's optimiser plan the work; that is why DataFrames are faster and why almost all modern PySpark code uses them. You may meet RDDs in older code or tutorials. On Databricks the RDD API only works on classic compute with **dedicated** (single-user) access mode; it is **not supported on serverless or standard (shared) compute**, which is what most workspaces use, so this session does not use it. If you ever see `.rdd` or `sparkContext.parallelize(...)`, read it as "the old way of doing what a DataFrame does".

<a id="lazy"></a>
# Section 6: Lazy evaluation: transformations and actions

**Definition.** Spark operations come in two kinds. **Transformations** (`select`, `filter`, `withColumn`, `groupBy`, `join`) describe a new DataFrame but do not compute it. **Actions** (`show`, `count`, `collect`, `toPandas`, writing a file) make Spark execute everything needed to produce a result. Waiting until an action to do any work is called **lazy evaluation**.

**Example.** `big = df.filter(F.col("amount") > 1000)` returns instantly, even on a billion rows. `big.count()` is the moment Spark reads data and filters it.

**Analogy.** Writing a recipe versus cooking it. Adding steps to the recipe costs nothing. Only when someone orders the dish (the action) does the kitchen start, and because the head chef reads the **whole** recipe first, they can reorganise it: chop all the onions at once, skip a step whose result is never used, or filter out bad ingredients before washing them.

**Explanation.** Lazy evaluation is what lets Spark **optimise**: it sees the complete chain of transformations before executing, so it can push filters down to the file reader, read only the columns needed, and combine steps into one pass. The practical consequences:

1. Errors in a transformation sometimes only appear when an action runs.
2. Each action re-runs the plan from the source unless the result is cached (notebook 02.04).
3. DataFrames are **immutable**: transformations return new DataFrames, so always reassign (`df = df.withColumn(...)`).

| Transformations (lazy) | Actions (trigger work) |
|---|---|
| `select`, `withColumn`, `drop` | `show()`, `count()` |
| `filter` / `where` | `collect()`, `take(n)`, `first()` |
| `groupBy(...).agg(...)` | `toPandas()` |
| `join`, `union`, `orderBy`, `distinct` | `write...` |

<a id="when"></a>
# Section 7: Spark or pandas?

Spark is not always the better tool. Starting a distributed job has overhead, so on small data pandas is often faster and simpler.

| Use pandas when... | Use Spark when... |
|---|---|
| The data fits comfortably in memory (up to a few GB) | The data is larger than one machine's memory |
| You need quick, interactive exploration | The same job must run reliably on growing volumes |
| You rely on libraries that expect pandas (many plotting and stats tools) | Data already lives in a data lake or on Databricks |
| The job runs in seconds anyway | The job is slow because it is single-threaded |

A very common pattern combines both: **do the heavy lifting in Spark** (filter, join, aggregate billions of rows down to thousands) and **bring the small result into pandas** for plotting or a final report with `toPandas()`.

<a id="first"></a>
# Section 8: Meet your Spark session

**Definition.** Every PySpark program talks to Spark through a **`SparkSession`**, the entry point that reads data, runs SQL and holds the configuration. On Databricks the session already exists in every notebook, under the name `spark`: there is nothing to install or start.

**Example.** `spark.version` tells you which Spark version your compute runs; `spark.conf.get("spark.sql.ansi.enabled")` tells you whether a bad type conversion raises an error or returns null.

**Analogy.** Getting into a hire car: the engine is already running (Databricks started Spark for you). Before driving off you glance at the dashboard: which model it is, and how a few settings are configured.

**Explanation.** `SparkSession.builder.appName(...).getOrCreate()` returns the existing session if there is one (on Databricks there always is) or starts a new one (on a laptop), so the same line works everywhere. `appName` labels the application in the Spark UI.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_01_02_IntroToPySpark").getOrCreate()
print("Spark version:", spark.version)

**The configuration.** Spark has hundreds of settings, read with `spark.conf.get(name)`. You will rarely change them, but a few explain behaviour you will see in this session:

| Setting | What it controls | Where it matters |
|---|---|---|
| `spark.sql.ansi.enabled` | `true`: a failed `cast` or `to_date` stops the query with an error. `false`: the value silently becomes null | Data cleaning (02.02): use `try_cast` on messy data |
| `spark.sql.shuffle.partitions` | How many pieces the data is split into after a `groupBy` or `join` (default 200) | Performance (02.04) |
| `spark.sql.adaptive.enabled` | Adaptive Query Execution: Spark adjusts the plan at run time from the real data sizes | Performance (02.04) |
| `spark.sql.autoBroadcastJoinThreshold` | Tables smaller than this are copied to every machine in a join instead of shuffled | Performance (02.04) |
| `spark.sql.session.timeZone` | The time zone used for timestamps | Dates and times |

On serverless compute some settings cannot be read or changed, so the cell below asks for each one separately and reports what it can.

In [ ]:
settings = [
    "spark.app.name",
    "spark.sql.ansi.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.session.timeZone",
]
for name in settings:
    try:
        value = spark.conf.get(name)
    except Exception:
        value = "(not readable on this compute)"
    print(f"{name:<40} {value}")

**Changing a setting for this session.** `spark.conf.set(name, value)` changes a setting until the session ends; it does not affect other users. On serverless only a few settings can be changed, among them `spark.sql.ansi.enabled`, `spark.sql.shuffle.partitions` and `spark.sql.session.timeZone` (Section 22 switches ANSI mode off for one cell).

**Where is the Spark UI?** The **Spark UI** is Spark's own web page: every job, stage and task your session ran, how long each took, and the full list of settings. Where you find it depends on the compute:

| Compute | Where to look |
|---|---|
| **Serverless** (the usual choice) | There is no Spark UI. Under the output of a cell, click **See performance** to list the queries the cell ran; click a query, then **See query profile** for its plan, timings and rows processed |
| **Classic compute** (a cluster) | Open **Compute** in the left sidebar, select your cluster, then the **Spark UI** tab |
| **A laptop** | `http://localhost:4040` in a browser, while the Spark session is running |

**A DataFrame from Python data.** A Spark **DataFrame** is a distributed table with named, typed columns. The quickest way to make one is from a list of tuples plus a schema (here written as a short `"name TYPE, ..."` string; Section 11 covers schemas).

In [ ]:
students = spark.createDataFrame(
    [(1, "Alice", "Math", 85), (2, "Bob", "Math", 78), (3, "Charlie", "Math", 92),
     (4, "Alice", "Science", 88), (5, "Bob", "Science", 74), (6, "Charlie", "Science", 95)],
    schema="id INT, student STRING, module STRING, score INT",
)
students.show()
students.printSchema()

**Lazy evaluation, seen.** The next cell defines a transformation and prints what it is: a DataFrame plan, with no data computed. The action on the following line produces the rows.

In [ ]:
top = students.filter(F.col("score") >= 85).select("student", "module", "score")
print(top)          # only a description of the result: no work done yet
top.show()          # the action: Spark now runs the filter and the select

#### Try it yourself

Using `students`, show only the Science results, sorted from highest to lowest score.

In [ ]:
# Your turn

In [ ]:
# Coach answer
students.filter(F.col("module") == "Science").orderBy(F.col("score").desc()).show()

#### Try it yourself

For each line below, decide whether it is a **transformation** or an **action**, then run them to check. Which lines produced output?

```python
a = students.select("student", "score")
b = a.filter(F.col("score") > 80)
b.count()
c = b.orderBy("score")
c.show(3)
```

In [ ]:
# Your turn

In [ ]:
# Coach answer
a = students.select("student", "score")      # transformation (no output)
b = a.filter(F.col("score") > 80)              # transformation (no output)
print(b.count())                               # action: runs select + filter
c = b.orderBy("score")                         # transformation (no output)
c.show(3)                                      # action: runs the whole chain again, plus the sort

# Part 2: The pandas walkthrough in PySpark

<a id="frompandas"></a>
# Section 9: From pandas to PySpark

You already know how to wrangle data in pandas. The goal of this notebook is that you can do **the same work in PySpark** without thinking twice, because on Databricks that is how data larger than one machine is handled.

| You want to... | pandas | PySpark |
|---|---|---|
| Read a CSV | `pd.read_csv(path)` | `spark.read.csv(path, header=True, inferSchema=True)` |
| Look at the first rows | `df.head()` | `df.show(5)` |
| Keep some columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| Keep some rows | `df[df["a"] > 0]` | `df.filter(F.col("a") > 0)` |
| Add a column | `df["c"] = df["a"] * 2` | `df = df.withColumn("c", F.col("a") * 2)` |
| Stack tables | `pd.concat([a, b])` | `a.unionByName(b)` |
| Merge tables | `pd.merge(a, b, on="k", how="left")` | `a.join(b, on="k", how="left")` |
| Summarise by group | `df.groupby("g").agg(...)` | `df.groupBy("g").agg(...)` |

Three differences explain most of the surprises you will meet:

1. **Immutability.** A Spark DataFrame never changes. Every operation returns a **new** DataFrame, so there is no `inplace=True`: you reassign (`df = df.withColumn(...)`).
2. **Laziness.** Transformations only build a plan; nothing runs until an action such as `show()` or `count()` (Section 6).
3. **No index.** The rows are spread across machines, so there is no row label like pandas' index, and no guaranteed order unless you sort.

<a id="setup"></a>
# Section 10: Reading data

**Definition.** `spark.read.csv(path, ...)` reads a CSV file (or a folder, or a wildcard of files) into a Spark **DataFrame**: a table with named, typed columns, split into partitions across the cluster.

**Example.** `customer = spark.read.csv(path, header=True, inferSchema=True)` is the twin of `customer = pd.read_csv(path)`.

**Analogy.** pandas reads the whole book into one person's memory. Spark hands chapters to a team of readers, each of whom only needs to hold their own chapters.

**Explanation.** Two options you will use on almost every read:

| Option | Meaning | pandas default |
|---|---|---|
| `header=True` | The first line holds the column names | pandas assumes this |
| `inferSchema=True` | Scan the data to guess each column's type; otherwise **every column is text** | pandas always guesses |

**How this course reads files.** Many Databricks computes (serverless included) are not allowed to read the course S3 bucket with `spark.read`. So every notebook reads with a small helper from the course `config.py`, the **pandas bridge**: `pandas_bridge(spark, name)` downloads the file with pandas over HTTPS and converts it into a Spark DataFrame. From then on it is an ordinary Spark DataFrame. Under each read, commented out, are two alternatives. The first is the bridge **written out in plain code**, `pd.read_csv(...)` then `spark.createDataFrame(...)`, so you can see there is no magic inside it. The second is the `spark.read.csv` version: that is how real big data is read, because every machine reads its own part of the file in parallel, whereas the bridge (and the plain version) passes the whole file through one machine.

> **Official documentation:** [`DataFrameReader.csv`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html) and the [CSV options](https://spark.apache.org/docs/latest/sql-data-sources-csv.html).

The setup cell in Section 8 already imported what we need: `F` holds Spark's built-in column functions (`F.col`, `F.sum`, `F.when`, ...) and `T` holds the data types.

**The customer table.** In pandas you wrote `customer = pd.read_csv(".../customer.csv")`. The read below follows the course pattern: the pandas bridge, with the Spark read as the commented alternative.

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
customer = pandas_bridge(spark, "customer")

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.read_csv(session_datasets_http["customer"])
# customer = spark.createDataFrame(pdf)

# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# customer = spark.read.csv(session_datasets["customer"], header=True, inferSchema=True)

In [ ]:
customer.show(5)

Notice the difference from pandas already: `customer` is not printed when you type its name; you get a short description of its columns. To see rows you ask for them with `show()`, an **action**.

In [ ]:
customer

The payment table is read the same way. (In the pandas walkthrough this was your first task; here it is done for you because the next sections build on it.)

In [ ]:
payment = pandas_bridge(spark, "payment")
# Alternative (plain pandas, then Spark): payment = spark.createDataFrame(pd.read_csv(session_datasets_http["payment"]))
# Alternative (Spark reads S3): payment = spark.read.csv(session_datasets["payment"], header=True, inferSchema=True)

#### Try it yourself

Show the first 5 rows of `payment` without truncating long values, and print how many payments there are.

In [ ]:
# Your turn

In [ ]:
# Coach answer
payment.show(5, truncate=False)
print("payments:", payment.count())

<a id="schema"></a>
# Section 11: Schemas: defining the structure of your data

**Definition.** A **schema** describes a DataFrame's structure: for each column, its **name**, its **data type**, and whether it may be empty (**nullable**). In Spark, defining the schema is how you **model** your data.

**Example.** The payment table: `payment_id` is a whole number, `amount` is money, `payment_date` is a date and time.

**Analogy.** An architect's plan: which rooms exist and what each is for. You can build without one, but surprises later are expensive.

**Explanation.** There are two ways to get a schema:

1. **Let Spark infer it.** Every read so far did this. `spark.read.csv(..., inferSchema=True)` scans the data and guesses each column's type (without `inferSchema`, every column is text). The pandas bridge does the same through pandas: pandas guesses a type for each column, and Spark maps it (`int64` becomes `bigint`, `float64` becomes `double`, text becomes `string`). `printSchema()` shows what was guessed.
2. **Declare it yourself**, as shown below.

Inference is convenient for exploration, but a guess can be wrong (below, `payment_date` arrives as text), and that is risky in a pipeline that runs every day:

| Problem with guessing | Example | With an explicit schema |
|---|---|---|
| Wrong type | US zip code `05408` read as a number becomes `5408` | Declare it as a string |
| Unstable type | One file holds `19140.0`, the next `19140`: a double one day, an integer the next | The type is fixed |
| Extra work | Inference reads the data an extra time | No extra pass |
| Money as floating point | Sums of amounts drift by tiny rounding errors | `DECIMAL(p, s)` stores exact values |

You write a schema in one of two equivalent ways: a **DDL string** (`"name TYPE, ..."`, short and readable) or a `StructType` of `StructField`s (explicit, handy when built in code).

**Syntax**

```python
schema = "payment_id INT, amount DECIMAL(5,2), payment_date TIMESTAMP"          # DDL string

schema = T.StructType([                                                         # StructType
    T.StructField("payment_id",   T.IntegerType(),      True),                  # True = nullable
    T.StructField("amount",       T.DecimalType(5, 2),  True),
    T.StructField("payment_date", T.TimestampType(),    True),
])

df = spark.read.csv(path, header=True, schema=schema)    # the order must match the file's columns
df = pandas_bridge(spark, name, schema=schema)           # the course helper takes the same argument
```

> **Official documentation:** [Data types](https://spark.apache.org/docs/latest/sql-ref-datatypes.html) and [`StructType`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.types.StructType.html).

In [ ]:
payment.printSchema()          # what the guess produced: payment_date is text, amount a double

In [ ]:
payment_schema = T.StructType([
    T.StructField("payment_id",   T.IntegerType(),     True),
    T.StructField("customer_id",  T.IntegerType(),     True),
    T.StructField("staff_id",     T.IntegerType(),     True),
    T.StructField("rental_id",    T.IntegerType(),     True),
    T.StructField("amount",       T.DecimalType(5, 2), True),    # money: exact decimals
    T.StructField("payment_date", T.TimestampType(),   True),
])

payment = pandas_bridge(spark, "payment", schema=payment_schema)
# Alternative (plain pandas, then Spark, then each column cast to the type in the schema):
# payment = spark.createDataFrame(pd.read_csv(session_datasets_http["payment"])).select(
#     [F.col(f.name).cast(f.dataType) for f in payment_schema.fields])
# Alternative (Spark reads S3): payment = spark.read.csv(session_datasets["payment"], header=True, schema=payment_schema)
payment.printSchema()
payment.show(3, truncate=False)

The same schema as a DDL string gives exactly the same structure:

In [ ]:
ddl = "payment_id INT, customer_id INT, staff_id INT, rental_id INT, amount DECIMAL(5,2), payment_date TIMESTAMP"
payment_ddl = pandas_bridge(spark, "payment", schema=ddl)
# Alternative (Spark reads S3): spark.read.csv(session_datasets["payment"], header=True, schema=ddl)
print("same schema:", payment_ddl.schema == payment.schema)
print(payment.schema.simpleString())

**The types you will use most:**

| Spark type | DDL name | Use for |
|---|---|---|
| `StringType` | `STRING` | Text, codes, ids and postcodes (anything with leading zeros) |
| `IntegerType` / `LongType` | `INT` / `BIGINT` | Whole numbers / very large whole numbers |
| `DoubleType` | `DOUBLE` | Measurements, ratios, model features |
| `DecimalType(p, s)` | `DECIMAL(p,s)` | **Money**: `p` digits in total, `s` after the point |
| `BooleanType` | `BOOLEAN` | Flags |
| `DateType` / `TimestampType` | `DATE` / `TIMESTAMP` | Dates / dates with times |

**Bad values.** If a value in the file does not fit the type (text in an `INT` column), the reader's `mode` option decides: `PERMISSIVE` (default) turns it into `null` and keeps the row, `DROPMALFORMED` drops the row, `FAILFAST` stops the read with an error. Data cleaning (notebook 02.02) shows how to find such values.

#### Try it yourself

Write a DDL schema for the customer table and read it again into `customer_typed`: `customer_id`, `store_id` and `address_id` as `INT`, the names and `email` as `STRING`, `activebool` as `BOOLEAN`, `create_date` as `DATE`, `last_update` as `TIMESTAMP` and `active` as `INT`. Print the schema.

> *Hint:* The column order in the schema must match the order in the file: check `customer.columns`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
customer_schema = ("customer_id INT, store_id INT, first_name STRING, last_name STRING, email STRING, "
                   "address_id INT, activebool BOOLEAN, create_date DATE, last_update TIMESTAMP, active INT")
customer_typed = pandas_bridge(spark, "customer", schema=customer_schema)
customer_typed.printSchema()

<a id="explore"></a>
# Section 12: Exploring the data

**Definition.** The first commands to run on any new DataFrame tell you its size, its columns and types, and what typical values look like.

**Example.** How many customers are there, which columns does the table have, and what are the ranges of the numeric columns?

**Analogy.** Walking round a house before buying it: count the rooms, check what each one is, look at the condition.

**Explanation.**

| pandas | PySpark | Notes |
|---|---|---|
| `df.head()` / `df.head(10)` | `df.show()` / `df.show(10)` | `show(truncate=False)` prints long values in full |
| `df.tail(2)` | `df.tail(2)` | Returns a list of `Row` objects |
| `df.columns` | `df.columns` | A Python list |
| `df.dtypes` | `df.dtypes` or `df.printSchema()` | |
| `df.info()` | `df.printSchema()` | |
| `len(df)` / `df.shape` | `df.count()` / `(df.count(), len(df.columns))` | `count()` is an action: Spark reads the data |
| `df.describe()` | `df.describe().show()` or `df.summary().show()` | `summary()` adds percentiles |

> **Official documentation:** [`DataFrame.printSchema`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.printSchema.html), [`DataFrame.describe`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.describe.html) and [`DataFrame.summary`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.summary.html).

In [ ]:
print("rows:", customer.count(), "| columns:", len(customer.columns))
print(customer.columns)
print(customer.dtypes)

In [ ]:
customer.printSchema()

In [ ]:
customer.describe("customer_id", "store_id", "active").show()

**Why does `.show()` have parentheses but `.columns` does not?** The same rule as in pandas. `columns` and `dtypes` are **attributes**: Spark already knows them from the schema, so nothing has to run. `show()`, `count()` and `describe()` are **methods** that do work: they make Spark read the data. In Spark that work can be large, which is why it waits until you ask.

In [ ]:
rows = customer.take(2)         # a list of Row objects, for use in Python code
print(rows[0])
print(rows[0]["first_name"], rows[0].last_name)

<a id="select"></a>
# Section 13: Selecting and renaming columns

**Definition.** `select` returns a new DataFrame with only the columns you list. **Aliases** give a column a new name in the result.

**Example.** Keep only the customers' first and last names; show `email` under the name `contact_email`.

**Analogy.** Choosing which columns of a spreadsheet to print: the full sheet still exists, you only see the part you asked for.

**Explanation.** There are four ways to refer to a column, and `select` accepts all of them. None needs SQL:

| Way | Example | When to use it |
|---|---|---|
| A string | `customer.select("first_name")` | The shortest; fine for plain selection |
| An attribute | `customer.select(customer.first_name)` | Like `df.col` in pandas; fails if the name has spaces |
| Brackets | `customer.select(customer["first_name"])` | Like `df["col"]` in pandas; works with any name |
| `F.col` | `customer.select(F.col("first_name"))` | Needed as soon as you **do something** with the column (maths, comparisons, `alias`) |

In pandas, `df["col"]` gives a Series and `df[["col"]]` gives a DataFrame. In Spark, `customer["first_name"]` or `F.col("first_name")` is a **Column**: an expression, not data. Only `select` turns it into a DataFrame.

In [ ]:
customer.select("first_name", "last_name").show(3)
customer.select(customer.first_name, customer["last_name"], F.col("email")).show(3, truncate=False)

In [ ]:
print(type(customer["first_name"]))          # a Column: a description of a column
print(type(customer.select("first_name")))     # a DataFrame

**Selecting from a list.** Because `columns` is a Python list, you can build the selection in code: the twin of `df[cols]` in pandas.

In [ ]:
name_cols = ["first_name", "last_name"]
customer.select(name_cols).show(2)

# every column except the timestamps
customer.select([c for c in customer.columns if c not in ("create_date", "last_update")]).show(2)

**Renaming with aliases, without SQL.** Four tools, from the most local to the most global:

| Tool | Renames | Example |
|---|---|---|
| `F.col("a").alias("b")` | A column **inside a `select`** | `customer.select(F.col("email").alias("contact_email"))` |
| `withColumnRenamed("a", "b")` | One column, keeping all the others | `customer.withColumnRenamed("email", "contact_email")` |
| `withColumnsRenamed({...})` | Several columns at once (Spark 3.4 and later) | `customer.withColumnsRenamed({"first_name": "first", "last_name": "last"})` |
| `toDF(*names)` | **Every** column, by position | `customer.toDF(*[c.upper() for c in customer.columns])` |

`alias` also names the result of a calculation, which otherwise gets an automatic name such as `(amount * 2)`. The pandas twin of `withColumnRenamed` is `df.rename(columns={"a": "b"})`.

> **Official documentation:** [`DataFrame.select`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html), [`Column.alias`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.alias.html), [`DataFrame.withColumnRenamed`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumnRenamed.html) and [`DataFrame.toDF`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toDF.html).

In [ ]:
customer.select(F.col("first_name").alias("first"), F.col("email").alias("contact_email")).show(2, truncate=False)

customer.withColumnRenamed("email", "contact_email").printSchema()

In [ ]:
# Several at once (Spark 3.4+), with a fallback for older versions
renames = {"first_name": "first", "last_name": "last"}
try:
    renamed = customer.withColumnsRenamed(renames)
except AttributeError:
    renamed = customer
    for old, new in renames.items():
        renamed = renamed.withColumnRenamed(old, new)
print(renamed.columns)

# Every column at once, by position. toDF expects the names as separate arguments, so the * in front
# of the list "unpacks" it: toDF(*["A", "B"]) is the same as toDF("A", "B")
print(customer.toDF(*[c.upper() for c in customer.columns]).columns)

**Unique values.** `select(...).distinct()` is the twin of pandas' `unique()`.

In [ ]:
customer.select("store_id").distinct().show()

#### Try it yourself

From `payment`, select `payment_id`, `amount` and `payment_date`, renaming `amount` to `amount_usd`. Show 5 rows.

In [ ]:
# Your turn

In [ ]:
# Coach answer
payment.select("payment_id", F.col("amount").alias("amount_usd"), "payment_date").show(5)

<a id="sort"></a>
# Section 14: Sorting, and why there is no index

**Definition.** `orderBy` (or its alias `sort`) returns the rows sorted by one or more columns; `limit(n)` keeps the first `n`.

**Example.** Customers sorted by first name; the stores and addresses in descending order.

**Analogy.** Sorting a pile of receipts. Easy for one person; on a cluster each machine holds part of the pile, so a full sort means passing receipts between machines, which is why sorting large data is expensive (notebook 02.04).

**Explanation.**

| pandas | PySpark |
|---|---|
| `df.sort_values(by="a")` | `df.orderBy("a")` |
| `df.sort_values(by=["a", "b"], ascending=False)` | `df.orderBy(F.desc("a"), F.desc("b"))` or `df.orderBy(["a", "b"], ascending=False)` |
| `df.sort_values(by="a", inplace=True)` | `df = df.orderBy("a")` (no `inplace`: reassign) |
| `df.nlargest(5, "a")` | `df.orderBy(F.desc("a")).limit(5)` |

> **Official documentation:** [`DataFrame.orderBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html) and [`functions.row_number`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html).

In [ ]:
customer.orderBy("first_name").select("customer_id", "first_name", "last_name").show(5)
customer.orderBy(F.desc("store_id"), F.desc("address_id")).select("customer_id", "store_id", "address_id").show(5)

**There is no index (and no `reset_index`).** In pandas every row has an index label, `.loc` and `.iloc` select by label or position, and after sorting you often call `reset_index()`. A Spark DataFrame has **none of these**: its rows are spread across machines and have no label and no fixed position. Two consequences:

1. **Order is only guaranteed right after `orderBy`.** If the order matters for a result, sort just before you look at it.
2. **To number rows, create a column.** `F.row_number()` over a sorted window gives 1, 2, 3, ... in the order you choose (window functions are covered in Section 20).

So "the 4th customer in reverse alphabetical order" is answered by sorting and taking the first four rows, rather than with `.loc` or `.iloc`.

In [ ]:
from pyspark.sql.window import Window

numbered = customer.withColumn("position", F.row_number().over(Window.orderBy(F.desc("first_name"), "customer_id")))
numbered.select("position", "first_name", "last_name").show(5)

fourth = customer.orderBy(F.desc("first_name"), "customer_id").take(4)[-1]
print("4th in reverse alphabetical order:", fourth["first_name"], fourth["last_name"])

A window with only `orderBy` puts every row in one partition, so Spark warns about performance. That is fine on 599 customers; on large data, number rows within groups (`Window.partitionBy(...)`) instead.

#### Try it yourself

Sort `payment` from the smallest to the largest `amount`, and from the largest `amount` down, showing `payment_id`, `customer_id` and `amount` (5 rows each). Save the descending version as `payment_sorted`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
payment.orderBy("amount").select("payment_id", "customer_id", "amount").show(5)
payment_sorted = payment.orderBy(F.desc("amount"))
payment_sorted.select("payment_id", "customer_id", "amount").show(5)

<a id="create"></a>
# Section 15: Creating columns, and the `.apply()` equivalents

**Definition.** `withColumn(name, expression)` returns a DataFrame with a new column, or with an existing column replaced. The expression is built from columns and the functions in `F`.

**Example.** A `username` made of first and last name; a payment amount doubled.

**Analogy.** A spreadsheet formula column: you write the rule once and it applies to every row, here in parallel on every partition.

**Explanation.**

| pandas | PySpark |
|---|---|
| `df["username"] = df["first_name"] + df["last_name"]` | `df = df.withColumn("username", F.concat("first_name", "last_name"))` |
| `df["empty"] = ""` | `df = df.withColumn("empty", F.lit(""))` (`F.lit` = a literal, the same value on every row) |
| `df["double"] = df["amount"] * 2` | `df = df.withColumn("double", F.col("amount") * 2)` |
| several at once | `df.withColumns({"a": ..., "b": ...})` (Spark 3.3 and later) |
| `df["number"] = [1, 2, 3, ...]` | Not possible: there is no index to line a Python list up with. Use `F.row_number()` (Section 14) or `F.monotonically_increasing_id()` (unique, not consecutive) |

The most useful building blocks in `F`: arithmetic on columns, `F.lit`, `F.concat` / `F.concat_ws` (join text), `F.upper` / `F.lower` / `F.length` / `F.trim` (text), `F.round`, `F.when(...).otherwise(...)` (if/else), `F.to_date` / `F.year` / `F.month` (dates).

> **Official documentation:** [`DataFrame.withColumn`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html) and the full [list of built-in functions (`F`)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html).

In [ ]:
customer = customer.withColumn("username", F.concat(F.col("first_name"), F.col("last_name")))
customer = customer.withColumn("empty", F.lit(""))
customer.select("first_name", "last_name", "username", "empty").show(3)

In [ ]:
payment.withColumn("double_amount", F.col("amount") * 2).select("payment_id", "amount", "double_amount").show(3)

In [ ]:
# A Python list cannot become a column; a generated id can
customer.withColumn("number", F.monotonically_increasing_id()).select("customer_id", "number").show(3)

#### Try it yourself

Create a column `half_price` in `payment` holding the amount divided by 2, rounded to 2 decimals, and show it next to `amount`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
payment.withColumn("half_price", F.round(F.col("amount") / 2, 2)).select("payment_id", "amount", "half_price").show(5)

### Applying functions: what replaces `.apply()`?

In the pandas walkthrough you wrote a Python function and applied it to every username:

```python
def name_length_category(name):
    if len(name) > 15:
        category = 'long'
    elif len(name) > 9:
        category = 'medium'
    else:
        category = 'short'
    return category

customer['name_length_category'] = customer['username'].apply(name_length_category)
# or, with a lambda:  customer['username'].apply(lambda n: 'long' if len(n) > 15 else ...)
```

**PySpark has no `.apply()` method on a DataFrame column.** You have three options, in order of preference:

| Option | How | Speed | When to use it |
|---|---|---|---|
| **1. Built-in functions** | `F.when(...).otherwise(...)`, `F.length`, `F.upper`, ... | Fast: runs inside Spark's engine | **The default.** Covers almost every `.apply()` you wrote in pandas |
| **2. A Python UDF** | Wrap your Python function (or a lambda) with `F.udf` | Slow: every row is sent to Python and back | Logic that built-in functions cannot express |
| **3. A pandas UDF** | `@F.pandas_udf` on a function that takes a pandas Series | Much faster than a plain UDF | Custom logic on large data |

**Analogy.** Built-in functions are the kitchen's own equipment: the cooks use it where they stand. A Python UDF is a dish only an outside caterer can make: every plate has to be carried out of the kitchen and back, one at a time.

> **Official documentation:**
> - Built-in functions: [`functions.when`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html) and the [full list of functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
> - Python UDFs: [`functions.udf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html) and the [PySpark guide to UDFs](https://spark.apache.org/docs/latest/api/python/user_guide/udfandudtf.html)
> - pandas UDFs: [`functions.pandas_udf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html) and [pandas UDFs with Apache Arrow](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_pandas.html)
> - On Databricks: [User-defined scalar functions in Python](https://docs.databricks.com/aws/en/udf/python)

**Option 1: built-in functions.** `when` conditions are checked in order and the first match wins, just like the `if / elif / else` in the function.

In [ ]:
customer = customer.withColumn(
    "name_length_category",
    F.when(F.length("username") > 15, "long")
     .when(F.length("username") > 9, "medium")
     .otherwise("short"),
)
customer.select("username", "name_length_category").show(5)
customer.groupBy("name_length_category").count().orderBy("name_length_category").show()

**Option 2: the same function as a UDF.** `F.udf(function, return_type)` turns a normal Python function, or a lambda, into something Spark can apply to a column. This is the closest thing to `.apply()`, and it gives the same answer. Some locked-down workspaces block Python UDFs, so the cell checks.

In [ ]:
def name_length_category(name):
    if len(name) > 15:
        return "long"
    elif len(name) > 9:
        return "medium"
    return "short"

name_length_udf = F.udf(name_length_category, T.StringType())                        # from a def
first_letter_udf = F.udf(lambda name: name[0] if name else None, T.StringType())     # from a lambda

try:
    (customer
        .withColumn("category_udf", name_length_udf("username"))
        .withColumn("initial", first_letter_udf("first_name"))
        .select("username", "name_length_category", "category_udf", "initial")
        .show(5))
except Exception as e:
    print("Python UDFs are not available on this compute:", type(e).__name__)

Both columns agree. So why prefer option 1? Spark cannot look inside a UDF, so it cannot optimise it, and every row has to be converted to a Python object and back. On millions of rows the built-in version is typically many times faster; notebook 02.04 measures it. Two further points:

- The **lambda** example also has a built-in twin: `F.substring("first_name", 1, 1)`. Look for one before writing a UDF.
- If you really want pandas syntax, `customer.pandas_api()` gives a **pandas API on Spark** DataFrame where `.apply()` exists, run by Spark underneath (Section 22). It is useful for porting pandas code, but the DataFrame API is the standard.

#### Try it yourself

Using built-in functions only, add a column `email_domain` to `customer` holding the part of the email after the `@` (hint: `F.split(col, "@")` returns an array; `F.element_at(array, 2)` takes its second element). How many different domains are there?

In [ ]:
# Your turn

In [ ]:
# Coach answer
with_domain = customer.withColumn("email_domain", F.element_at(F.split("email", "@"), 2))
with_domain.select("email", "email_domain").show(3, truncate=False)
print("distinct domains:", with_domain.select("email_domain").distinct().count())

<a id="drop"></a>
# Section 16: Removing columns

**Definition.** `drop("a", "b")` returns a DataFrame without the listed columns.

**Example.** Remove the helper columns `empty` and `username`.

**Analogy.** Hiding columns in a printout; the original data is untouched.

**Explanation.**

| pandas | PySpark |
|---|---|
| `df.drop("a", axis=1)` | `df.drop("a")` (always columns: rows are removed with `filter`) |
| `df.drop("a", axis=1, inplace=True)` | `df = df.drop("a")` |
| `del df["a"]` | No equivalent: DataFrames are immutable, so reassign with `drop` |
| `df.drop(["a", "b"], axis=1)` | `df.drop("a", "b")` or `df.drop(*cols)` for a list |

**Why the `*` in `drop(*cols)`?** `drop` takes the column names as **separate arguments**, `drop("a", "b")`, not as one list. Putting `*` in front of a list **unpacks** it into separate arguments, so `drop(*["a", "b"])` is exactly `drop("a", "b")`. It is plain Python (the same `*` you can use with any function), handy when the names come from code.

Dropping a column that does not exist is not an error in Spark: it is silently ignored, so check the column names when a drop seems to have no effect.

> **Official documentation:** [`DataFrame.drop`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.drop.html).

In [ ]:
print(customer.columns)
customer = customer.drop("empty", "username")
print(customer.columns)

#### Try it yourself

Create `payment_extra` from `payment` with the two columns `double_amount` (amount times 2) and `half_price` (amount divided by 2), then drop both in a single call using a Python list.

In [ ]:
# Your turn

In [ ]:
# Coach answer
payment_extra = payment.withColumn("double_amount", F.col("amount") * 2).withColumn("half_price", F.col("amount") / 2)
extra_cols = ["double_amount", "half_price"]
payment_extra = payment_extra.drop(*extra_cols)     # the * unpacks the list: same as drop("double_amount", "half_price")
print(payment_extra.columns)

<a id="filter"></a>
# Section 17: Filtering rows

**Definition.** `filter(condition)` (or its alias `where`) keeps only the rows for which the condition is true.

**Example.** Customers of store 2; customers called Willie; payments under 1 dollar.

**Analogy.** A sieve: matching rows pass through. Spark applies the sieve as early as it can, even while reading the file, so fewer rows travel through the rest of the plan.

**Explanation.** In pandas you wrote `customer[customer["store_id"] == 2]`. In PySpark the condition is a Column expression, and there are three equivalent ways to write it, plus `where` as an alias of `filter`:

| Way | Example |
|---|---|
| `F.col` | `customer.filter(F.col("store_id") == 2)` |
| Attribute | `customer.filter(customer.store_id == 2)` |
| Brackets | `customer.filter(customer["store_id"] == 2)` |
| `where` (alias) | `customer.where(F.col("store_id") == 2)` |

All four produce the same plan. `F.col` is the most flexible because it does not need the DataFrame's name, so it keeps working inside long chains.

> **Official documentation:** [`DataFrame.filter`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html), [`Column.isin`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.isin.html) and [`Column.between`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.between.html).

In [ ]:
print(customer.filter(F.col("store_id") == 2).count())
print(customer.filter(customer.store_id == 2).count())
print(customer.filter(customer["store_id"] == 2).count())
print(customer.where(F.col("store_id") == 2).count())

**Breaking it down.** As in pandas, the condition on its own is not a list of True/False values but an expression; you can store it and reuse it.

In [ ]:
store_2 = F.col("store_id") == 2
willie = F.upper(F.col("first_name")) == "WILLIE"
print(store_2)

customer.filter(willie).select("customer_id", "first_name", "last_name", "store_id").show()

**Combining conditions.** Use `&` (and), `|` (or) and `~` (not), and **wrap every condition in parentheses**. Python's `and` / `or` do not work on Spark columns: they raise `ValueError: Cannot convert column into bool`.

| Need | Operator or method | Example |
|---|---|---|
| And | `&` | `(F.col("store_id") == 2) & willie` |
| Or | the pipe operator | `(F.col("amount") < 1)`, pipe, `(F.col("amount") > 10)` |
| Not | `~` | `~(F.col("store_id") == 2)` |
| One of several values | `isin` | `F.col("staff_id").isin(1, 2)` |
| A range (inclusive) | `between` | `F.col("amount").between(5, 10)` |
| Missing / present | `isNull` / `isNotNull` | `F.col("email").isNull()` (never `== None`) |
| Text | `startswith`, `endswith`, `contains`, `like` | `F.col("last_name").startswith("MC")` |

In [ ]:
customer.filter(store_2 & willie).select("customer_id", "first_name", "last_name", "store_id").show()
customer.filter((F.col("store_id") == 2) & (F.upper(F.col("first_name")) == "WILLIE")).count()

In [ ]:
payment.filter((F.col("amount") < 1) | (F.col("amount") > 10)).count()

In [ ]:
customer.filter(F.col("last_name").startswith("MC")).select("first_name", "last_name").show()
payment.filter(F.col("amount").between(9, 10)).select("payment_id", "amount").show(3)

The classic mistake, on purpose (caught so you can read the message):

In [ ]:
try:
    payment.filter(F.col("amount") < 1 or F.col("amount") > 10).show()
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])
    print("Fix: use | (or &) and wrap each condition in parentheses.")

**Rows and columns together: the `.loc` equivalent.** pandas' `customer.loc[condition, ["customer_id", "email"]]` becomes a `filter` followed by a `select`. Positional selection (`.iloc`) has no equivalent, because rows have no position (Section 14).

> **A SQL-flavoured alternative.** `filter` also accepts a SQL condition as a string: `customer.filter("store_id = 2 AND first_name = 'WILLIE'")`. It produces the same plan. This notebook uses column expressions, which are safer when conditions are built in code.

In [ ]:
customer.filter(willie).select("customer_id", "email", "active").show(truncate=False)

#### Try it yourself

How many payments are **less than 1 dollar**? And how many are less than 1 **or** greater than 10? Write each filter in a different one of the three styles (`F.col`, attribute, brackets).

In [ ]:
# Your turn

In [ ]:
# Coach answer
print("under 1:", payment.filter(F.col("amount") < 1).count())
print("under 1 or over 10:", payment.filter((payment.amount < 1) | (payment["amount"] > 10)).count())

<a id="write"></a>
# Section 18: Writing data to files

**Definition.** `df.write` saves a DataFrame. Unlike `to_csv` in pandas, Spark writes a **folder** containing one part file per partition, because each machine writes its own piece at the same time.

**Example.** Save the customers of store 2 as CSV, next to this notebook.

**Analogy.** A team writing a report: each member hands in their own chapter (the part files), and the folder is the report.

**Explanation.**

| pandas | PySpark |
|---|---|
| `df.to_csv("out.csv", index=False)` | `df.write.csv("out_folder", header=True, mode="overwrite")` |
| one file | `df.coalesce(1).write.csv(...)`: a single part file (small results only) |
| `df.to_parquet(...)` | `df.write.parquet(...)` (columnar, compressed, keeps the schema) |
| `df.to_excel(...)` | No built-in writer: `df.toPandas().to_excel(...)` for small results |

`mode` controls what happens if the folder exists: `"overwrite"`, `"append"`, `"ignore"` or `"error"` (the default).

**On Databricks** there are two places to save:

| Where | How | For |
|---|---|---|
| **The notebook's own folder** (the default below) | A file name with no folder, e.g. `"customers_store_2.csv"`, written with pandas: `df.toPandas().to_csv(...)` | Small results you want to download or keep next to the notebook |
| **A Unity Catalog Volume** | Spark's writer with a path like `/Volumes/<catalog>/<schema>/<volume>/...` | Large results, written in parallel as a folder of part files (Spark cannot write into the notebook's folder) |

The standard format for tables on Databricks is **Delta** (`df.write.format("delta").saveAsTable(...)`).

> **Write permissions.** In a shared training workspace you may not be allowed to write files. The cell below catches the error, so the rest of the notebook still runs.

> **Official documentation:** [`DataFrameWriter.csv`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.csv.html), [Databricks: work with files](https://docs.databricks.com/aws/en/files/) and [Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes/).

In [ ]:
store_2_customers = customer.filter(F.col("store_id") == 2).select("customer_id", "first_name", "last_name", "email")

# Default: a small result saved as ONE CSV file in the same folder as this notebook.
# A file name with no folder in front of it means "next to the notebook".
OUTPUT_FILE = "customers_store_2.csv"
try:
    store_2_customers.toPandas().to_csv(OUTPUT_FILE, index=False)    # toPandas(): only for small results
    print("saved", os.path.abspath(OUTPUT_FILE))
except OSError as e:
    print("Could not write a file here ->", type(e).__name__, "(training workspaces often do not allow it)")

# The Spark way, for large results: a FOLDER of part files, written by every machine in parallel.
# On Databricks it must go to a Unity Catalog Volume (ask your administrator which one you can use):
# store_2_customers.write.csv("/Volumes/<catalog>/<schema>/<volume>/customers_store_2", header=True, mode="overwrite")

<a id="multiple"></a>
# Section 19: Working with multiple datasets: unions and joins

The Superstore orders arrive as **one file per year**, and the sales representatives in a separate file. As in Session 2, we first **stack** the three years (`pd.concat` in pandas, a **union** in Spark), then **merge** in the representatives (`pd.merge`, a **join**).

**Reading the orders.** The first column of these files has an empty header: it is the pandas index saved by mistake. The pandas bridge passes `index_col=0` to pandas, which uses that column as the index so it never reaches Spark. (With the Spark read, the column arrives as `_c0` and is dropped; Spark also needs `escape='"'`, because product names contain quotes written as `""`.)

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
orders_2017 = pandas_bridge(spark, "orders_2017", index_col=0)

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.read_csv(session_datasets_http["orders_2017"], index_col=0)
# orders_2017 = spark.createDataFrame(pdf)

# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# orders_2017 = spark.read.csv(session_datasets["orders_2017"], header=True, inferSchema=True, escape='"')

orders_2017 = orders_2017.drop("_c0")      # only needed after the Spark read: the saved pandas index
orders_2017.show(3)

In [ ]:
orders_2018 = pandas_bridge(spark, "orders_2018", index_col=0)
orders_2019 = pandas_bridge(spark, "orders_2019", index_col=0)
# Alternative (plain pandas, then Spark), for example:
# orders_2018 = spark.createDataFrame(pd.read_csv(session_datasets_http["orders_2018"], index_col=0))
# Alternative (Spark reads S3), for example:
# orders_2018 = spark.read.csv(session_datasets["orders_2018"], header=True, inferSchema=True, escape='"').drop("_c0")

print(orders_2017.count(), orders_2018.count(), orders_2019.count())

### Unions: stacking DataFrames

**Definition.** A **union** appends the rows of one DataFrame below another, like `pd.concat([a, b])`. It keeps duplicates (like SQL `UNION ALL`); add `.distinct()` to remove them.

**Analogy.** Putting this year's pages behind last year's in a binder. It only makes sense if the pages have the same layout.

**Explanation.** There are two versions, and the difference matters:

| Method | Matches columns by | Risk |
|---|---|---|
| `a.union(b)` | **Position** (1st with 1st, 2nd with 2nd) | Silently mixes data if the columns are in a different order |
| `a.unionByName(b)` | **Name** | Fails loudly if the names differ, which is what you want |

pandas' `concat` matches by **name**, so `unionByName` is its true twin. Compare the column names of the three files:

In [ ]:
for name, df in [("2017", orders_2017), ("2018", orders_2018), ("2019", orders_2019)]:
    print(name, [c for c in df.columns if c in ("dhip_date", "ship_date", "country_segment", "county_region", "country_region")])

Two typos in 2017 (`dhip_date`, `country_segment`) and one in 2018 (`county_region`). `unionByName` refuses to stack them, because the names do not match:

In [ ]:
try:
    orders_2017.unionByName(orders_2018).count()
except Exception as e:
    print("unionByName refused:", type(e).__name__)

`union` would accept them, because it only looks at positions. Here the columns happen to be in the same order, so the data would land in the right place, but the result would keep the 2017 names (with the typos) and you would not have been warned. The safe route is the pandas one: **rename, then stack by name**.

In [ ]:
orders_2017 = orders_2017.withColumnRenamed("dhip_date", "ship_date").withColumnRenamed("country_segment", "country_region")
orders_2018 = orders_2018.withColumnRenamed("county_region", "country_region")

orders = orders_2017.unionByName(orders_2018).unionByName(orders_2019)
print("rows:", orders.count())       # 2,102 + 2,587 + 3,312
orders.select("order_id", "order_date", "ship_date", "country_region", "region", "sales", "profit").show(3)

`unionByName(other, allowMissingColumns=True)` also stacks files where one has an extra column, filling the gap with nulls. And when files share the same layout and names, a single **wildcard** read (`spark.read.csv(".../orders_*.csv")`) unions them as it reads.

> **Official documentation:** [`DataFrame.union`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.union.html) and [`DataFrame.unionByName`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unionByName.html).

### Joins: merging DataFrames

**Definition.** A **join** puts two DataFrames side by side, pairing each row of the left table with the rows of the right table that have the same value in the **join key**. The join type (`how`) decides what happens to rows with no partner.

**Example.** Add the sales representative of each region to every order.

**Analogy.** Matching name badges to a guest list by ID. The join type is the door policy: only guests with a badge (inner), every guest with or without a badge (left), or everyone from both lists (full).

**Explanation.**

| pandas | PySpark |
|---|---|
| `pd.merge(left, right, on="k", how="left")` | `left.join(right, on="k", how="left")` |
| `left_on="a", right_on="b"` | `left.join(right, left["a"] == right["b"], "left")` |

| `how` | Keeps | Typical use |
|---|---|---|
| `"inner"` (default) | Rows with a match on both sides | Only matched records |
| `"left"` | All left rows; unmatched get nulls | Enrich a table without losing rows |
| `"right"` / `"full"` | All right rows / all rows from both | Reconciliations |
| `"left_semi"` | Left rows that **have** a match (no columns added) | "Keep only customers who paid" |
| `"left_anti"` | Left rows with **no** match | Find orphans and gaps |

**Two habits that prevent silent errors:** count the rows before and after a join (a join can multiply rows if the key repeats on the right), and select only the columns you need from the right table (columns with the same name on both sides become ambiguous).

> **Official documentation:** [`DataFrame.join`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html).

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
reps = pandas_bridge(spark, "sales_reps", index_col=0)

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.read_csv(session_datasets_http["sales_reps"], index_col=0)
# reps = spark.createDataFrame(pdf)

# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# reps = spark.read.csv(session_datasets["sales_reps"], header=True, inferSchema=True)

reps = reps.drop("_c0")      # only needed after the Spark read
reps.show()

In [ ]:
order_reps = orders.join(reps, on="region", how="left")

print("orders:", orders.count(), "| after the join:", order_reps.count())
order_reps.select("order_id", "region", "person", "sales").show(5)

# Any region without a representative? A left anti join lists them
print("orders whose region has no rep:", orders.join(reps, "region", "left_anti").count())

**When the key has a different name on each side.** In pandas you wrote `pd.merge(left, right, left_on="region", right_on="rep_region")`. PySpark has two ways to do the same:

| Way | Code | Result |
|---|---|---|
| **1. An explicit join condition** | `left.join(right, left["region"] == right["rep_region"], "left")` | **Both** key columns are kept: drop the copy afterwards |
| **2. Rename first, then join on the shared name** | `left.join(right.withColumnRenamed("rep_region", "region"), on="region", how="left")` | One key column, as with `on=` |

To try it, we make a copy of `reps` whose key is called `rep_region`, as it might be in a file from another team.

In [ ]:
reps_other = reps.withColumnRenamed("region", "rep_region")        # the key now has a different name
print(reps_other.columns)

# Way 1: an explicit condition (the twin of left_on= / right_on=)
way_1 = orders.join(reps_other, orders["region"] == reps_other["rep_region"], how="left")
way_1.select("order_id", "region", "rep_region", "person").show(3)    # both key columns are there
way_1 = way_1.drop("rep_region")                                       # keep one of them

# Way 2: rename the key so both sides match, then join on the common name
way_2 = orders.join(reps_other.withColumnRenamed("rep_region", "region"), on="region", how="left")

print("rows:", orders.count(), way_1.count(), way_2.count())          # the same: no rows gained or lost

#### Try it yourself

Join `payment` to `customer` to show each payment with the customer's first and last name (keep every payment). Select `payment_id`, `first_name`, `last_name` and `amount`, and confirm the row count did not change.

In [ ]:
# Your turn

In [ ]:
# Coach answer
pay_names = payment.join(customer.select("customer_id", "first_name", "last_name"), on="customer_id", how="left")
pay_names.select("payment_id", "first_name", "last_name", "amount").show(5)
print(payment.count(), pay_names.count())

<a id="agg"></a>
# Section 20: Aggregating data

**Definition.** An **aggregation** reduces many rows to summary values (sum, mean, count, ...). `groupBy` first splits the rows into groups, then each group is aggregated.

**Example.** Total sales, quantity and profit per product category.

**Analogy.** Sorting receipts into one envelope per category and writing the total on each. On a cluster each machine totals its own receipts per category first, and only those partial totals are exchanged.

**Preparing the data.** As in pandas, the columns must have the right types first. `profit` is text such as `"-$383.03"` or `"$1,234.56"`, so we strip the `$` and the thousands separator with `F.regexp_replace` and convert it with `try_cast`, which gives `null` instead of an error if a value cannot be converted (Section 8, ANSI mode). The dates become real dates with `cast("date")`.

In [ ]:
order_reps.select("order_date", "discount", "profit").printSchema()

order_reps = (
    order_reps
    .withColumn("profit", F.regexp_replace("profit", "[$,]", ""))          # "-$1,234.56" -> "-1234.56"
    .withColumn("profit", F.expr("try_cast(profit AS DOUBLE)"))
    .withColumn("order_date", F.col("order_date").cast("date"))
    .withColumn("ship_date", F.col("ship_date").cast("date"))
)
order_reps.select("order_date", "discount", "profit").printSchema()
print("profit values that could not be converted:", order_reps.filter(F.col("profit").isNull()).count())

**Syntax.**

| pandas | PySpark |
|---|---|
| `df.groupby("g").sum(numeric_only=True)` | `df.groupBy("g").sum("a", "b")` (name the columns) |
| `df.groupby("g").agg({"a": "sum", "b": "mean"})` | `df.groupBy("g").agg({"a": "sum", "b": "mean"})` (same dictionary form) |
| named results | `df.groupBy("g").agg(F.sum("a").alias("total_a"), F.avg("b").alias("avg_b"))` |
| `df.groupby(["g", "h"])` | `df.groupBy("g", "h")` |
| `df["a"].sum()` (whole column) | `df.agg(F.sum("a")).first()[0]` |

Unlike pandas, Spark never tries to sum text columns: you list exactly what to aggregate. Prefer the `F` form with `alias`, otherwise Spark invents names such as `sum(sales)`.

> **Official documentation:** [`DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html), [`GroupedData.agg`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.agg.html) and [window functions (`Window`)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html).

In [ ]:
order_reps.groupBy("category").sum("sales", "quantity", "discount", "profit").orderBy("category").show()

In [ ]:
(
    order_reps.groupBy("category")
    .agg(
        F.round(F.sum("sales"), 2).alias("sales"),
        F.sum("quantity").alias("quantity"),
        F.round(F.avg("discount"), 3).alias("avg_discount"),
        F.round(F.sum("profit"), 2).alias("profit"),
    )
    .orderBy("category")
    .show()
)

The dictionary form, as in pandas, and grouping by two columns:

In [ ]:
order_reps.groupBy("category").agg({"sales": "sum", "quantity": "sum", "discount": "mean", "profit": "sum"}).show()

(
    order_reps.groupBy("category", "sub_category")
    .agg(F.round(F.sum("sales"), 2).alias("sales"), F.round(F.sum("profit"), 2).alias("profit"))
    .orderBy("category", F.desc("profit"))
    .show(8)
)

**Whole-column aggregations** (pandas' `df["sales"].sum()`): `agg` without `groupBy`. `.first()[0]` brings the single value back into Python.

In [ ]:
totals = order_reps.agg(
    F.round(F.sum("sales"), 2).alias("total_sales"),
    F.round(F.avg("profit"), 2).alias("avg_profit"),
    F.min("quantity").alias("min_qty"),
    F.max("quantity").alias("max_qty"),
)
totals.show()
print("total sales as a Python number:", totals.first()["total_sales"])

**Filtering after aggregating** (SQL's `HAVING`) is simply a `filter` on the aggregated DataFrame. The ten customers with the highest sales:

In [ ]:
(
    order_reps.groupBy("customer_id")
    .agg(F.round(F.sum("sales"), 2).alias("sales"))
    .orderBy(F.desc("sales"))
    .limit(10)
    .show()
)

**Optional: ranking within groups (window functions).** A window function computes a value for each row from a group of related rows **without** collapsing them, as `groupBy` would. The best-selling sub-category in each category:

In [ ]:
sub = order_reps.groupBy("category", "sub_category").agg(F.round(F.sum("sales"), 2).alias("sales"))
w = Window.partitionBy("category").orderBy(F.desc("sales"))
sub.withColumn("rank", F.row_number().over(w)).filter(F.col("rank") == 1).show()

<a id="sql"></a>
# Section 21: The same in Spark SQL

**Definition.** Spark SQL lets you query a DataFrame with ordinary SQL. `createOrReplaceTempView("name")` gives the DataFrame a name SQL can use; `spark.sql("...")` runs a query and returns a DataFrame.

**Example.** Total profit by region, in SQL.

**Analogy.** Two languages describing the same recipe: Spark translates both into the same kitchen instructions.

**Explanation.** SQL and the DataFrame API compile to the **same plan**, so there is no performance difference. Use SQL when the logic is naturally a query, and the DataFrame API when you need Python around it (loops, functions, parameters). On Databricks you can also start a cell with `%sql` and write pure SQL.

> **Official documentation:** [`SparkSession.sql`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.sql.html) and the [Spark SQL reference](https://spark.apache.org/docs/latest/sql-ref.html).

In [ ]:
order_reps.createOrReplaceTempView("order_reps")

spark.sql("""
    SELECT region, ROUND(SUM(profit), 2) AS profit
    FROM order_reps
    GROUP BY region
    ORDER BY profit DESC
""").show()

In [ ]:
# The same query with the DataFrame API
order_reps.groupBy("region").agg(F.round(F.sum("profit"), 2).alias("profit")).orderBy(F.desc("profit")).show()

**Translation table.**

| SQL | DataFrame API |
|---|---|
| `SELECT a, b AS c` | `.select("a", F.col("b").alias("c"))` |
| `WHERE a > 1 AND b = 'x'` | `.filter((F.col("a") > 1) & (F.col("b") == "x"))` |
| `GROUP BY g` | `.groupBy("g").agg(...)` |
| `HAVING SUM(a) > 10` | `.filter(...)` after `agg` |
| `ORDER BY a DESC LIMIT 5` | `.orderBy(F.desc("a")).limit(5)` |
| `CASE WHEN ... THEN ... ELSE ... END` | `F.when(...).otherwise(...)` |
| `JOIN t ON ...` / `UNION ALL` | `.join(t, ...)` / `.unionByName(t)` |

<a id="pandasapi"></a>
# Section 22: The pandas API on Spark

**Definition.** The **pandas API on Spark** (the module `pyspark.pandas`, formerly a project called Koalas) lets you write **pandas syntax** that Spark runs across the cluster. `df.pandas_api()` turns a Spark DataFrame into a pandas-on-Spark DataFrame, `.to_spark()` turns it back, and `.to_pandas()` brings it into real pandas on one machine.

**Example.** `psdf.groupby("region")["profit"].sum()` is the pandas line you already know, but the data stays spread across the cluster.

**Analogy.** A translator. You speak pandas, and the translator turns each sentence into Spark instructions. Very convenient, but some expressions do not translate exactly, and a translation is never quite as precise as speaking the language yourself.

**Explanation.** It is most useful for **porting existing pandas code** to data that no longer fits in memory, with few changes. Three cautions:

| Caution | Why |
|---|---|
| Not everything behaves exactly like pandas | There is no real index across machines, so anything that depends on row positions (`iloc`, unsorted `head`) can be slow or give a different row order |
| ANSI mode | On Spark 4, and on Databricks serverless where ANSI mode is on, it refuses to run (`PANDAS_API_ON_SPARK_FAIL_ON_ANSI_MODE`). Switch ANSI mode off for the cells that use it, as below |
| It is not the standard | Most PySpark code, documentation and examples use the DataFrame API you learned in this notebook |

```python
import pyspark.pandas as ps
psdf = sdf.pandas_api()      # Spark DataFrame  -> pandas-on-Spark DataFrame
sdf = psdf.to_spark()        # pandas-on-Spark  -> Spark DataFrame
pdf = psdf.to_pandas()       # pandas-on-Spark  -> real pandas (all rows to one machine: small results only)
```

> **Official documentation:** [Pandas API on Spark](https://spark.apache.org/docs/latest/api/python/tutorial/pandas_on_spark/index.html) and [Databricks: Pandas API on Spark](https://docs.databricks.com/aws/en/pandas/pandas-on-spark).

In [ ]:
import pyspark.pandas as ps

# The pandas API on Spark refuses to run while ANSI mode is on, so remember the current
# setting, switch ANSI mode off for this cell, and always put it back at the end (finally).
try:
    ansi_before = spark.conf.get("spark.sql.ansi.enabled")
except Exception:
    ansi_before = "true"
spark.conf.set("spark.sql.ansi.enabled", "false")
try:
    psdf = order_reps.pandas_api()                       # Spark DataFrame -> pandas-on-Spark DataFrame
    print(type(psdf))

    # From here on it is pandas syntax, but Spark does the work
    print(psdf[["order_id", "region", "sales", "profit"]].head(3))
    print(psdf.groupby("region")["profit"].sum().sort_values(ascending=False).round(2))
except Exception as e:
    print("The pandas API on Spark is not available on this compute:", type(e).__name__)
finally:
    spark.conf.set("spark.sql.ansi.enabled", ansi_before)   # back to the original setting

The profit by region matches Section 21: same data, same engine, a third way of writing it. When you finish porting, `psdf.to_spark()` gives you an ordinary Spark DataFrame again.

#### Try it yourself

With the pandas API on Spark, count the order lines per `category` in `order_reps` using `value_counts()`. Remember to switch ANSI mode off for the cell and back on at the end.

In [ ]:
# Your turn

In [ ]:
# Coach answer
spark.conf.set("spark.sql.ansi.enabled", "false")      # the pandas API on Spark needs ANSI mode off
try:
    print(order_reps.pandas_api()["category"].value_counts())
finally:
    spark.conf.set("spark.sql.ansi.enabled", ansi_before)   # restore the original setting

<a id="application"></a>
# Section 23: Application: a sales report from three years of orders

**Scenario.** These are the questions the Session 2 walkthrough asked of the merged orders. Answer them in PySpark with `order_reps`, the union of the three years joined to the representatives, with `profit` converted to a number.

**Question 1: which sales representative has the highest average sale?**

In [ ]:
(
    order_reps.groupBy("person")
    .agg(F.round(F.avg("sales"), 2).alias("avg_sales"), F.count("*").alias("order_lines"))
    .orderBy(F.desc("avg_sales"))
    .show()
)

**Question 2: which region generated the highest total profit across 2017 to 2019?**

In [ ]:
order_reps.groupBy("region").agg(F.round(F.sum("profit"), 2).alias("total_profit")).orderBy(F.desc("total_profit")).show()

**Question 3: which day generated the highest sales?**

In [ ]:
order_reps.groupBy("order_date").agg(F.round(F.sum("sales"), 2).alias("sales")).orderBy(F.desc("sales")).show(3)

**Question 4: profit margin per year and category**, brought into pandas to plot (the "aggregate in Spark, plot in pandas" pattern: only 9 rows leave Spark).

In [ ]:
import matplotlib.pyplot as plt

margins = (
    order_reps.withColumn("year", F.year("order_date"))
    .groupBy("year", "category")
    .agg(F.round(100 * F.sum("profit") / F.sum("sales"), 1).alias("margin_pct"))
    .orderBy("year", "category")
)
margins.show()

pdf = margins.toPandas().pivot(index="year", columns="category", values="margin_pct")
pdf.plot.bar(figsize=(8, 4), rot=0)
plt.ylabel("profit margin (%)")
plt.title("Superstore: profit margin by year and category")
plt.tight_layout()
plt.show()

**Reading the result.** Chuck Magee (East) has the highest average sale, the West is the most profitable region, and the best single day was 2 October 2018. Technology and Office Supplies keep 16% to 21% of their sales as profit every year, while Furniture keeps under 4%: it sells as much as the other categories but earns very little on it.

<a id="exercises"></a>
# Exercises

The tasks from the Session 2 walkthrough, in PySpark. Use `orders_2017` (with the corrected column names) unless stated.

### Exercise 1: Types and a preview

Print the data types of `orders_2017`, then show the first 10 rows of its first 5 columns.

> *Hint:* `orders_2017.columns[:5]` is a Python list of the first five names.

In [ ]:
# Your turn

In [ ]:
# Coach answer
print(orders_2017.dtypes)
orders_2017.select(orders_2017.columns[:5]).show(10)

### Exercise 2: One order

Show the order date and the sales amount for the order with `order_id` `"CA-2017-153549"`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
orders_2017.filter(F.col("order_id") == "CA-2017-153549").select("order_id", "order_date", "sales").show()

### Exercise 3: The biggest sale

Without changing `orders_2017`, find the order line with the highest sales.

In [ ]:
# Your turn

In [ ]:
# Coach answer
orders_2017.orderBy(F.desc("sales")).select("order_id", "product_name", "sales").show(1, truncate=False)

### Exercise 4: Average item price

Add a column `avg_item_price` (sales divided by quantity). How many order lines have an average item price below 20? Then drop the column again.

In [ ]:
# Your turn

In [ ]:
# Coach answer
with_avg = orders_2017.withColumn("avg_item_price", F.col("sales") / F.col("quantity"))
print("lines with avg item price < 20:", with_avg.filter(F.col("avg_item_price") < 20).count())
with_avg = with_avg.drop("avg_item_price")
print("avg_item_price" in with_avg.columns)

<a id="additional"></a>
# Additional Exercises

Use `order_reps`.

### Additional Exercise 1: Region within category

How do sales and profit vary across regions within each product category? Show the totals rounded to 2 decimals, sorted by category then profit.

In [ ]:
# Your turn

In [ ]:
# Coach answer
(
    order_reps.groupBy("category", "region")
    .agg(F.round(F.sum("sales"), 2).alias("sales"), F.round(F.sum("profit"), 2).alias("profit"))
    .orderBy("category", F.desc("profit"))
    .show(12)
)

### Additional Exercise 2: Shipping modes by region

For each `ship_mode` and `region`, show the number of order lines, total sales and the average number of days between order and shipping.

> *Hint:* `F.datediff(end, start)` returns the number of days between two dates.

In [ ]:
# Your turn

In [ ]:
# Coach answer
(
    order_reps.withColumn("days_to_ship", F.datediff("ship_date", "order_date"))
    .groupBy("ship_mode", "region")
    .agg(F.count("*").alias("lines"), F.round(F.sum("sales"), 2).alias("sales"), F.round(F.avg("days_to_ship"), 1).alias("avg_days"))
    .orderBy("ship_mode", "region")
    .show(16)
)

### Additional Exercise 3: Discounts per segment

For each customer `segment`, show total sales and the minimum, maximum and average discount.

In [ ]:
# Your turn

In [ ]:
# Coach answer
(
    order_reps.groupBy("segment")
    .agg(F.round(F.sum("sales"), 2).alias("sales"),
         F.min("discount").alias("min_discount"), F.max("discount").alias("max_discount"),
         F.round(F.avg("discount"), 3).alias("avg_discount"))
    .show()
)

<a id="challenge"></a>
# Challenge

Back to the Pagila tables: management wants to know who the best customers are and how the two stores compare.

### Challenge part 1: Best customers

Compute the total paid and the number of payments per customer, join the customers' names, and show the five customers who paid the most.

In [ ]:
# Your turn

In [ ]:
# Coach answer
per_customer = payment.groupBy("customer_id").agg(F.sum("amount").alias("total_paid"), F.count("*").alias("payments"))
top5 = (
    per_customer.join(customer.select("customer_id", "first_name", "last_name", "store_id"), "customer_id", "left")
    .orderBy(F.desc("total_paid"))
    .limit(5)
)
top5.show()

### Challenge part 2: Stores compared

Using the customer's `store_id`, compare the two stores: total paid, number of payments, average payment (2 decimals) and number of distinct paying customers.

In [ ]:
# Your turn

In [ ]:
# Coach answer
(
    payment.join(customer.select("customer_id", "store_id"), "customer_id", "inner")
    .groupBy("store_id")
    .agg(F.sum("amount").alias("total_paid"), F.count("*").alias("payments"),
         F.round(F.avg("amount"), 2).alias("avg_payment"), F.countDistinct("customer_id").alias("customers"))
    .orderBy("store_id")
    .show()
)

### Challenge part 3: Monthly trend

Show the total paid per month (`yyyy-MM`) for each store, as one row per month and one column per store (`pivot`).

> *Hint:* `groupBy("month").pivot("store_id").agg(...)` turns each store into a column.

In [ ]:
# Your turn

In [ ]:
# Coach answer
(
    payment.join(customer.select("customer_id", "store_id"), "customer_id", "inner")
    .withColumn("month", F.date_format("payment_date", "yyyy-MM"))
    .groupBy("month").pivot("store_id").agg(F.sum("amount"))
    .orderBy("month")
    .show()
)
# Payments run from 24 January to 14 May 2007, so January and May are partial months.

<a id="takeaways"></a>
## Key Takeaways

**Spark and PySpark**

| Concept | In one line |
|---|---|
| Apache Spark | A distributed engine that processes data in parallel across a cluster, in memory where possible |
| PySpark | Spark's Python API; Python describes the work, the JVM engine does it |
| Driver / executors | The driver plans and coordinates; executors hold the data and run tasks |
| Partition / task / shuffle | A chunk of data / the work on one chunk / moving rows between executors (the expensive part) |
| Transformation vs action | Transformations build a plan lazily; actions execute it, which lets Spark optimise the whole plan |
| SparkSession and configuration | On Databricks `spark` already exists; `spark.version`, `spark.conf.get(name)`, `spark.conf.set(name, value)` |
| RDD | The old low-level API under DataFrames; not supported on serverless or standard compute, not used here |
| Spark vs pandas | Spark for data too big or jobs too slow for one machine; aggregate in Spark, then `toPandas()` the small result |

**From pandas to PySpark**

| Task | pandas | PySpark |
|---|---|---|
| Read a CSV | `pd.read_csv(path)` | `spark.read.csv(path, header=True, inferSchema=True)` or `schema=...`; in this course `pandas_bridge(spark, name)`, which is `spark.createDataFrame(pd.read_csv(url))` plus a tidy-up |
| Define types | `dtype=` / `astype` | A schema: `"a INT, b DECIMAL(10,2), c DATE"` or `StructType` |
| First rows / types | `head()`, `dtypes`, `info()` | `show()`, `dtypes`, `printSchema()` |
| Size | `len(df)`, `shape` | `count()`, `len(df.columns)` |
| Select columns | `df[["a", "b"]]` | `select("a", "b")`, `df.a`, `df["a"]`, `F.col("a")` |
| Rename | `rename(columns={...})` | `alias`, `withColumnRenamed`, `withColumnsRenamed`, `toDF` |
| Sort | `sort_values` | `orderBy("a")`, `orderBy(F.desc("a"))` |
| Index | `reset_index`, `loc`, `iloc` | No index: sort, then `limit` / `take`; number rows with `row_number` |
| New column | `df["c"] = ...` | `df = df.withColumn("c", ...)` |
| `.apply(func)` | `df["a"].apply(f)` | Built-ins (`F.when`, `F.length`, ...) first; `F.udf(f)` only if needed |
| Drop columns | `drop(..., axis=1)`, `del` | `drop("a", "b")` |
| Filter rows | `df[cond]`, `loc` | `filter(cond)` / `where(cond)` with `&`, `~` and the pipe; parentheses around each condition |
| Save | `to_csv` | Small: `toPandas().to_csv("file.csv")` next to the notebook; large: `write.csv(volume_folder, header=True, mode="overwrite")` |
| Stack | `pd.concat` | `unionByName` (by name; `union` is by position) |
| Merge | `pd.merge(a, b, on=, how=)` | `a.join(b, on=, how=)`, plus `left_semi` / `left_anti` |
| Merge on different names | `left_on=`, `right_on=` | `a.join(b, a["x"] == b["y"])`, or rename first |
| Aggregate | `groupby().agg()` | `groupBy().agg(F.sum(...).alias(...))` |
| SQL | n/a | `createOrReplaceTempView` + `spark.sql(...)` |
| pandas syntax on Spark | (it is pandas) | `df.pandas_api()`, back with `.to_spark()`; switch ANSI mode off for those cells |

<a id="conclusion"></a>
## Conclusion

You now know what Spark is, how its driver and executors share the work, why it waits until an action to compute anything, how to read your session's configuration, and when Spark is worth using. You have also done the whole Session 2 walkthrough in PySpark: reading files with a proper schema, exploring, selecting and renaming, sorting without an index, creating columns with built-in functions (and knowing when a UDF is the last resort), filtering, writing, stacking, joining and aggregating, in both the DataFrame API and SQL. The practical (notebook 01.03) puts these steps together on five Pagila tables to find the best-selling EU countries.

<a id="reading"></a>
## Further Reading & Resources

- Apache Spark: [Project home](https://spark.apache.org/)
- Apache Spark: [Cluster mode overview (driver, executors, cluster manager)](https://spark.apache.org/docs/latest/cluster-overview.html)
- Apache Spark: [PySpark DataFrame quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html)
- Apache Spark: [Spark SQL, DataFrames and Datasets guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- Apache Spark: [Spark configuration reference](https://spark.apache.org/docs/latest/configuration.html)
- Apache Spark: [RDD programming guide (background reading)](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
- Databricks: [Serverless compute limitations (RDD APIs not supported)](https://docs.databricks.com/aws/en/compute/serverless/limitations)
- Databricks: [Develop code in Databricks notebooks](https://docs.databricks.com/aws/en/notebooks/notebooks-code)
- Databricks: [Debugging with the Spark UI (classic compute)](https://docs.databricks.com/aws/en/compute/troubleshooting/debugging-spark-ui)
- Databricks: [Query profile (serverless)](https://docs.databricks.com/aws/en/sql/user/queries/query-profile)
- Apache Spark: [PySpark SQL functions reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- Apache Spark: [CSV data source options (header, schema, escape, mode)](https://spark.apache.org/docs/latest/sql-data-sources-csv.html)
- Apache Spark: [Data types reference](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)
- Apache Spark: [`F.udf`: Python user-defined functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html)
- Apache Spark: [PySpark user guide: Python UDFs and UDTFs](https://spark.apache.org/docs/latest/api/python/user_guide/udfandudtf.html)
- Databricks: [User-defined scalar functions in Python](https://docs.databricks.com/aws/en/udf/python)
- Apache Spark: [pandas UDFs and Apache Arrow](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_pandas.html)
- Apache Spark: [Pandas API on Spark](https://spark.apache.org/docs/latest/api/python/tutorial/pandas_on_spark/index.html)
- Databricks: [Pandas API on Spark](https://docs.databricks.com/aws/en/pandas/pandas-on-spark)
- Apache Spark: [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)